# LeetCode #170 — Two Sum III: Data Structure Design

**Vị trí 38 trong danh sách Easy — học thiết kế dữ liệu có trạng thái bằng Python.**

Trang bài: https://leetcode.com/problems/two-sum-iii-data-structure-design/ (**Premium / có thể bị khóa**). Mô tả, chữ ký và ví dụ được đối chiếu với https://leetcode.doocs.org/en/lc/170/ .

**Khác biệt quan trọng:** bài này **không nộp `class Solution`**. LeetCode yêu cầu **một lớp `TwoSum` với hai method**:

```python
class TwoSum:
    def __init__(self):
        ...
    def add(self, number: int) -> None:
        ...
    def find(self, value: int) -> bool:
        ...
```

Kết thúc notebook, bạn sẽ tự hiểu:
1. **Đối tượng (object)**, `class`, `self`, `__init__`, phương thức `add` / `find` và **trạng thái được lưu lại** là gì.
2. Vì sao bài này khác Two Sum (#1) chỉ gọi một lần trên một mảng có sẵn.
3. Cách lưu trong `list` (brute force), sau đó tối ưu bằng **dictionary đếm tần suất**.
4. Lý do phải phân biệt `x != y` với `x == y`, nếu không sẽ **dùng một số đúng một lần thành hai lần**.
5. Cách viết trace nhiều thao tác, code submit, test và 6 bài tập từ dễ đến khó.

**Hướng dẫn:** đừng nhảy đến code tối ưu. Ở bài này, phần khó không phải phép cộng, mà là **đối tượng phải nhớ những lần `add` trước đó**.

## 1. Đề bài bằng tiếng Việt dễ hiểu

Bạn cần thiết kế một “hộp số” ban đầu **rỗng**. Người dùng có thể gọi hai thao tác **nhiều lần, theo bất kỳ thứ tự nào**:

- `add(number)`: bỏ thêm **một** số nguyên vào hộp, không trả ra kết quả hữu ích (`None`).
- `find(value)`: hỏi **hiện tại** trong hộp có **hai phần tử ở hai lượt thêm khác nhau** có tổng đúng bằng `value` không? Trả `True` hoặc `False`.

Ví dụ theo đề:

```text
hộp ban đầu: []
add(1)          -> hộp [1]
add(3)          -> hộp [1, 3]
add(5)          -> hộp [1, 3, 5]
find(4)         -> True, vì 1 + 3 = 4
find(7)         -> False, vì không có cặp nào cộng được 7
```

**Lưu ý:** Một số có thể được `add` nhiều lần, ví dụ `add(2)` rồi `add(2)` → hai bản sao khác nhau, khi đó `find(4)` là `True`.

Theo mô tả tham khảo #170: `-10^5 <= number <= 10^5`, `value` trong miền số nguyên 32 bit, và tổng cộng tối đa `10^4` lần gọi `add` / `find`. Những test mở rộng bên dưới vẫn dùng giá trị hợp lệ.

In [1]:
# Chỉ mô phỏng bằng một list để thấy hộp được thêm dữ liệu như thế nào.
box = []
box.append(1)
box.append(3)
box.append(5)
print("Hộp hiện tại:", box)
print("Có cặp tổng 4?", (1 in box) and (3 in box))
# Dòng trên CHỈ là minh họa, chưa phải hàm find đúng tổng quát.

Hộp hiện tại: [1, 3, 5]
Có cặp tổng 4? True


## 2. Phân biệt Two Sum (#1) và Two Sum III (#170)

Bài **Two Sum (#1)** nhận ngay cả mảng `nums` và `target`, thường yêu cầu trả **chỉ số** hai phần tử. Ví dụ `[2,7,11,15], target=9` → `[0,1]`.

Bài **Two Sum III (#170)** không đưa sẵn mảng; các phần tử đến **từng lần** qua `add`, rồi người dùng có thể `find` khi nào cũng được. `find` chỉ trả **True/False**. Không trả chỉ số.

Điểm mới: phải xây một **data structure (cấu trúc dữ liệu)** có khả năng **ghi nhớ** trạng thái để dùng cho những lần gọi sau.

Hãy tự hỏi: nếu gọi `find(4)` trước khi thêm số 3 thì kết quả phải khác lúc đã thêm 3 không? **Có!** Vì nội dung của hộp đã thay đổi.

## 3. Nền tảng Python: `list`, `append`, `in` và `bool`

- `numbers = []`: tạo danh sách rỗng.
- `numbers.append(2)`: thêm **một bản sao** của số `2` vào cuối.
- `x in numbers`: kiểm tra có phần tử có giá trị bằng `x` không, kết quả kiểu `bool` (`True` / `False`).
- `len(numbers)`: số phần tử, **tính cả các bản sao trùng nhau**.

Ví dụ `[2,2]` có `len == 2`, nhưng `set([2,2])` chỉ có một giá trị phân biệt là `{2}`. Với bài này, **hai bản sao là hai phần tử thật sự** — không được vô tình xóa trùng lặp.

In [2]:
numbers = []
numbers.append(2)
print("Sau lần thêm 1:", numbers, "| len:", len(numbers))
numbers.append(2)
print("Sau lần thêm 2:", numbers, "| len:", len(numbers))
print("2 in numbers:", 2 in numbers)
print("Số lần xuất hiện của 2:", numbers.count(2))
print("Số giá trị phân biệt:", len(set(numbers)))

Sau lần thêm 1: [2] | len: 1
Sau lần thêm 2: [2, 2] | len: 2
2 in numbers: True
Số lần xuất hiện của 2: 2
Số giá trị phân biệt: 1


## 4. Nền tảng Python: `class` là gì? `object` là gì?

Hãy tưởng tượng **`class` là bản thiết kế hộp**, còn **object (đối tượng) là một chiếc hộp cụ thể** được tạo ra từ bản thiết kế đó.

```python
class Box:
    def __init__(self):
        self.numbers = []
```

- `class Box:` khai báo kiểu hộp.
- `Box()` **tạo một đối tượng mới**.
- `__init__` (constructor/khởi tạo) chạy khi tạo đối tượng.
- `self` chỉ **đúng đối tượng đang thao tác**.
- `self.numbers = []` là dữ liệu **bên trong mỗi chiếc hộp**, tồn tại qua nhiều lần gọi method.

Nếu tạo `a = Box()` và `b = Box()` thì `a.numbers` và `b.numbers` là **hai list độc lập**. Thêm vào hộp a không làm hộp b thay đổi.

In [3]:
class Box:
    def __init__(self):
        self.numbers = []

a = Box()
b = Box()
a.numbers.append(7)
print("a.numbers:", a.numbers)
print("b.numbers:", b.numbers)
assert a.numbers == [7] and b.numbers == []

a.numbers: [7]
b.numbers: []


## 5. Python cơ bản: method, dấu chấm và `self`

Nếu một hàm nằm **trong class**, ta gọi nó là **method (phương thức)**.

Giả sử có `def add(self, number)`, thì:

```python
obj.add(5)
```

nghĩa là: gọi method `add` của **đối tượng `obj`**, truyền giá trị `number = 5`; Python tự truyền `self = obj` khi gọi theo cú pháp này.

**Tại sao không thể chỉ dùng biến cục bộ `numbers = []` trong mỗi lần `add`?** Vì biến ấy được tạo mới mỗi lần gọi, rồi mất đi; dữ liệu của các lượt trước không được nhớ. Ta cần `self.numbers`.

In [4]:
class TinyBox:
    def __init__(self):
        self.numbers = []

    def add(self, number: int) -> None:
        self.numbers.append(number)

obj = TinyBox()
obj.add(1)
obj.add(3)
print("Trạng thái sau hai lượt add:", obj.numbers)
print("Giá trị trả về của add:", obj.add(5))
print("Trạng thái sau lượt thứ ba:", obj.numbers)

Trạng thái sau hai lượt add: [1, 3]
Giá trị trả về của add: None
Trạng thái sau lượt thứ ba: [1, 3, 5]


## 6. `None`, `True`, `False` và type hint

- `None` nghĩa là **không có giá trị trả về hữu ích**. Nếu method không có `return ...`, Python mặc định trả `None`.
- `True` / `False` là hai giá trị boolean dùng để trả lời câu hỏi có/không.
- `def add(self, number: int) -> None`: gợi ý rằng `number` là số nguyên, còn method không cần trả kết quả.
- `def find(self, value: int) -> bool`: gợi ý trả boolean.

**Type hint** (gợi ý kiểu) giúp người đọc hiểu API. Python không tự cấm mọi giá trị khác kiểu chỉ vì ta khai báo type hint.

`find(...)` cần **`return`**. Chỉ `print(True)` sẽ không trả True cho LeetCode.

In [5]:
def print_true_only():
    print(True)

def return_true():
    return True

print("Gọi print_true_only():")
result_1 = print_true_only()
print("Giá trị trả về:", result_1)
result_2 = return_true()
print("return_true() trả về:", result_2, type(result_2).__name__)

Gọi print_true_only():
True
Giá trị trả về: None
return_true() trả về: True bool


## 7. Cách 1 — Brute Force bằng một `list`

Xuất phát từ điều dễ nghĩ nhất:

- `__init__`: tạo `self.numbers = []`.
- `add(number)`: `self.numbers.append(number)`.
- `find(value)`: thử **mọi cặp vị trí** `(i,j)` với `i < j`; nếu `numbers[i] + numbers[j] == value`, trả `True`; nếu thử hết không thấy, trả `False`.

**Vì sao bắt đầu `j = i + 1`?** Để chắc chắn chọn **hai phần tử ở hai vị trí khác nhau**, không dùng lại cùng một phần tử hai lần.

Ví dụ `[2]` thì không có cặp để tổng bằng 4. Còn `[2,2]` có cặp hai vị trí 0 và 1.

In [6]:
class TwoSumListBrute:
    def __init__(self):
        self.numbers = []

    def add(self, number: int) -> None:
        self.numbers.append(number)

    def find(self, value: int) -> bool:
        n = len(self.numbers)
        for i in range(n):
            for j in range(i + 1, n):
                if self.numbers[i] + self.numbers[j] == value:
                    return True
        return False

box = TwoSumListBrute()
box.add(1)
box.add(3)
box.add(5)
print("find(4):", box.find(4))
print("find(7):", box.find(7))

find(4): True
find(7): False


## 8. Mổ xẻ `range`, chỉ số `i`, `j` và vòng lặp lồng nhau

`range(n)` tạo các chỉ số `0,1,...,n-1`.

`range(i+1,n)` tạo các chỉ số **sau `i`**, không có `i`.

Với 3 phần tử `[1,3,5]`, các cặp được kiểm đúng một lần:

| i | j | Cặp giá trị | Tổng |
|---:|---:|---|---:|
| 0 | 1 | 1 và 3 | 4 |
| 0 | 2 | 1 và 5 | 6 |
| 1 | 2 | 3 và 5 | 8 |

Không có `(0,0)`, không có `(1,0)`, không có việc dùng một phần tử hai lần. Đây là lý do `j` bắt đầu ở `i + 1`.

In [7]:
numbers = [1, 3, 5]
for i in range(len(numbers)):
    for j in range(i + 1, len(numbers)):
        print(f"i={i}, j={j}, {numbers[i]} + {numbers[j]} = {numbers[i] + numbers[j]}")

i=0, j=1, 1 + 3 = 4
i=0, j=2, 1 + 5 = 6
i=1, j=2, 3 + 5 = 8


## 9. Test brute force: điều gì xảy ra giữa các lượt gọi?

Với `box`, sau `add(2)` một lần, `find(4)` phải False. Sau khi `add(2)` thêm lần nữa, `find(4)` mới True.

Cũng cần test rằng `find` **không sửa nội dung hộp**; query chỉ kiểm tra chứ không thêm/xóa giá trị.

Với `add(-1)` và `add(3)`, `find(2)` phải True, vì `-1 + 3 = 2`.

In [8]:
box = TwoSumListBrute()
assert box.find(4) is False
box.add(2)
assert box.find(4) is False
box.add(2)
assert box.find(4) is True
assert box.numbers == [2, 2]
box.add(-1)
box.add(3)
assert box.find(2) is True
assert box.numbers == [2, 2, -1, 3]
print("PASS: hộp rỗng, bản sao trùng nhau, số âm, find không sửa dữ liệu")

PASS: hộp rỗng, bản sao trùng nhau, số âm, find không sửa dữ liệu


## 10. Nút thắt: tại sao cách list chậm nếu gọi `find` liên tục?

Gọi `n` là **tổng số lần add**, tính cả số trùng. `find` brute force trong trường hợp xấu nhất thử `n(n-1)/2` cặp:

- `n=10` có tối đa `45` cặp.
- `n=100` có tối đa `4.950` cặp.
- `n=10.000` có tối đa gần `50 triệu` cặp cho **một lần `find`**.

Trong khi đó, `add` chỉ `append` → `O(1)` **trung bình / amortized**.

| Cách list | Thời gian | Bộ nhớ |
|---|---|---|
| `add` | `O(1)` amortized | Thêm một số |
| `find` | `O(n²)` worst case | `O(1)` phụ mỗi lần gọi |
| Toàn đối tượng | — | `O(n)` |

Muốn `find` nhanh hơn, ta cần tránh thử mọi cặp. Ý tưởng quen thuộc từ Two Sum (#1): biết `x`, hãy tính **phần còn thiếu**.

In [9]:
for n in [10, 100, 1000, 10000]:
    pairs = n * (n - 1) // 2
    print(f"n={n:>5} -> tối đa {pairs:>10} cặp cần xem")

n=   10 -> tối đa         45 cặp cần xem
n=  100 -> tối đa       4950 cặp cần xem
n= 1000 -> tối đa     499500 cặp cần xem
n=10000 -> tối đa   49995000 cặp cần xem


## 11. Bước đột phá: tìm `y = value - x`

Cần biết liệu có hai số `x` và `y` sao cho:

```text
x + y = value
```

Nếu đã chọn `x`, ta **không cần duyệt từng `y`**: suy ngay `y = value - x`.

Ví dụ muốn tổng `10` và đang xét `x = 3`, ta cần `y = 7`. Nếu hộp có số 7, cặp này hợp lệ (ngoại trừ trường hợp x và y bằng nhau cần kiểm số bản sao).

Để tra cứu nhanh `y` có trong hộp không, ta dùng **Hash Map / dictionary**. Nhưng thay vì chỉ ghi `x` có/không, bài này còn phải biết **đã add x mấy lần**.

In [10]:
for value, x in [(10, 3), (4, 2), (0, -3), (7, 1)]:
    needed = value - x
    print(f"Muốn {value}, đã có x={x}, cần y={needed}")

Muốn 10, đã có x=3, cần y=7
Muốn 4, đã có x=2, cần y=2
Muốn 0, đã có x=-3, cần y=3
Muốn 7, đã có x=1, cần y=6


## 12. Nền tảng dictionary: **key → tần suất**

Hãy lưu:

```python
counts = {1: 1, 3: 1, 5: 1}
```

Nó có nghĩa là đã thêm một số 1, một số 3, một số 5.

Nếu `add(3)` thêm lần nữa, thành `{1:1, 3:2, 5:1}`.

- `counts = {}`: hộp ban đầu trống.
- `counts.get(number, 0)`: số lần gặp `number` trước đó; mặc định 0.
- `counts[number] = counts.get(number,0)+1`: thêm đúng một bản sao.
- `number in counts`: kiểm tra số có trong hộp.

Ta **không cần danh sách vị trí** nếu `find` chỉ trả bool; nhưng rất cần **số lượng bản sao** để xử lý `2+2`.

In [11]:
counts = {}
for x in [1, 3, 5, 3]:
    before = counts.get(x, 0)
    counts[x] = before + 1
    print(f"add({x}): trước={before} -> sau={counts[x]}; counts={counts}")
print("Có số 7?", 7 in counts)

add(1): trước=0 -> sau=1; counts={1: 1}
add(3): trước=0 -> sau=1; counts={1: 1, 3: 1}
add(5): trước=0 -> sau=1; counts={1: 1, 3: 1, 5: 1}
add(3): trước=1 -> sau=2; counts={1: 1, 3: 2, 5: 1}
Có số 7? False


## 13. Python `.items()` là gì?

Khi viết `for x, count in counts.items():`, ta đọc từng **cặp khóa và giá trị** trong dictionary:

```text
x = giá trị số đã từng add
count = số lần số x đã được add
```

`counts.items()` không có nghĩa là “vị trí index”. Với `{2:3, 7:1}`, nó cung cấp hai cặp `(2,3)` và `(7,1)`.

Trong `find`, chỉ cần duyệt **các giá trị phân biệt**, nên số lượt lặp là `k` (số khóa), có thể nhỏ hơn rất nhiều so với tổng số lần `add` là `n`.

In [12]:
counts = {2: 3, 7: 1}
for x, count in counts.items():
    print("Số đã thêm:", x, "| đã thêm bao nhiêu lần:", count)
print("Số khóa phân biệt:", len(counts))

Số đã thêm: 2 | đã thêm bao nhiêu lần: 3
Số đã thêm: 7 | đã thêm bao nhiêu lần: 1
Số khóa phân biệt: 2


## 14. Vấn đề cực quan trọng: khi `x == y` thì sao?

Cần tìm hai **lần thêm khác nhau**, không phải cùng một số dùng hai lần.

Ví dụ 1:

```text
add(2)
find(4) -> False
```

Vì chỉ có **một bản sao** của số 2; không được lấy chính nó hai lần.

Ví dụ 2:

```text
add(2)
add(2)
find(4) -> True
```

Vì đã có **hai bản sao khác nhau**.

Vậy khi tính `y = value - x`:

- Nếu `x != y`: chỉ cần `y in counts` (vì `x` chắc chắn đã có: ta đang duyệt khóa x).
- Nếu `x == y`: cần **`counts[x] >= 2`**, tức ít nhất hai lần `add(x)`.

Đây là điểm dễ sai nhất của bài #170.

In [13]:
for data in [{2: 1}, {2: 2}, {2: 3}, {1: 1, 3: 1}]:
    value = 4
    possible = False
    for x, frequency in data.items():
        y = value - x
        if y in data and (x != y or frequency >= 2):
            possible = True
            break
    print("counts=", data, "| find(4) ->", possible)

counts= {2: 1} | find(4) -> False
counts= {2: 2} | find(4) -> True
counts= {2: 3} | find(4) -> True
counts= {1: 1, 3: 1} | find(4) -> True


## 15. Vì sao dùng `set` thôi là chưa đủ?

`set` chỉ nhớ **một giá trị có xuất hiện hay không**: thêm 2 hai lần vào một set thì vẫn chỉ có `{2}`.

Vì thế, nếu bạn chỉ lưu `seen = {2}` rồi kiểm `needed in seen`, có nguy cơ nhầm `2 + 2 = 4` là hợp lệ dù chưa từng thêm số 2 lần thứ hai.

Dùng **dictionary đếm tần suất** khắc phục được chuyện này: `{2: 1}` và `{2: 2}` là hai trạng thái khác nhau.

Bạn cũng có thể dùng `Counter` trong Python, nhưng hiểu `dict` trước sẽ giúp bạn tự viết lại được trong phỏng vấn.

In [14]:
s = set()
s.add(2)
s.add(2)
print("set sau hai lần add(2):", s)
counts = {}
counts[2] = counts.get(2, 0) + 1
counts[2] = counts.get(2, 0) + 1
print("dict sau hai lần add(2):", counts)

set sau hai lần add(2): {2}
dict sau hai lần add(2): {2: 2}


## 16. Cách 2 — Thiết kế `TwoSum` bằng Hash Map

Chia công việc ra từng method:

**`__init__`**: tạo `self.counts = {}` một lần khi tạo đối tượng.

**`add(number)`**: cập nhật `self.counts[number]` thêm 1.

**`find(value)`**: với mỗi khóa `x` trong dictionary:

1. Tính `y = value - x`.
2. Nếu `y` không có trong dictionary, thử `x` khác.
3. Nếu `y` có và `x != y`, đã có **hai giá trị khác nhau** → `True`.
4. Nếu `x == y`, phải kiểm số lần `add(x)` ít nhất 2.
5. Nếu không tìm thấy cặp nào, `False`.

**Invariant của đối tượng:** tại mọi thời điểm, `self.counts[x]` đúng bằng **số lần `add(x)` đã gọi trên đối tượng đó**.

In [15]:
class TwoSumHashMap:
    def __init__(self):
        self.counts = {}

    def add(self, number: int) -> None:
        self.counts[number] = self.counts.get(number, 0) + 1

    def find(self, value: int) -> bool:
        for x, frequency in self.counts.items():
            y = value - x
            if y in self.counts:
                if x != y or frequency >= 2:
                    return True
        return False

box = TwoSumHashMap()
for number in [1, 3, 5]:
    box.add(number)
print("counts:", box.counts)
print("find(4):", box.find(4))
print("find(7):", box.find(7))

counts: {1: 1, 3: 1, 5: 1}
find(4): True
find(7): False


## 17. Đọc từng dòng điều kiện `if` thật chậm

```python
if y in self.counts:
    if x != y or frequency >= 2:
        return True
```

`or` nghĩa là **ít nhất một trong hai điều kiện đúng**.

| x | y | `x != y` | `frequency >= 2` | Có thể nhận cặp? |
|---:|---:|:---:|:---:|:---:|
| 1 | 3 | True | Không quan trọng | Có nếu 3 tồn tại |
| 2 | 2 | False | False khi chỉ có một số 2 | Không |
| 2 | 2 | False | True khi có hai số 2 | Có |

**Dòng ngoài** `if y in ...` là bắt buộc: nếu `y` chưa từng được thêm, không thể có cặp, dù `x != y` là True.

Có thể viết gọn trong **một** điều kiện, nhưng khi mới học nên tách 2 bước cho rõ ràng.

In [16]:
cases = [(1, 3, 1, True), (2, 2, 1, False), (2, 2, 2, True)]
for x, y, frequency, expected in cases:
    good_when_both_exist = (x != y or frequency >= 2)
    print(f"x={x}, y={y}, count[x]={frequency} -> điều kiện phụ: {good_when_both_exist}")
    assert good_when_both_exist == expected

x=1, y=3, count[x]=1 -> điều kiện phụ: True
x=2, y=2, count[x]=1 -> điều kiện phụ: False
x=2, y=2, count[x]=2 -> điều kiện phụ: True


## 18. Bảng trace bằng tay: một chuỗi thao tác `add` / `find`

| Bước | Thao tác | `counts` trước | Hành động | `counts` sau / Kết quả |
|---:|---|---|---|---|
| 1 | `add(1)` | `{}` | Tăng đếm 1 | `{1:1}` |
| 2 | `add(3)` | `{1:1}` | Tăng đếm 3 | `{1:1,3:1}` |
| 3 | `find(4)` | `{1:1,3:1}` | Với x=1, cần y=3 có trong hộp | **True**, không đổi counts |
| 4 | `find(6)` | `{1:1,3:1}` | Cặp 3+3 cần 2 bản sao, mới có 1 | **False**, không đổi counts |
| 5 | `add(3)` | `{1:1,3:1}` | Tăng đếm 3 | `{1:1,3:2}` |
| 6 | `find(6)` | `{1:1,3:2}` | 3+3 hợp lệ vì có 2 bản sao | **True**, counts không đổi |

Hãy chú ý: **một câu hỏi `find(6)` có thể thay đổi câu trả lời từ False sang True sau một lượt `add(3)`**. Đó là biểu hiện của một đối tượng có trạng thái.

In [17]:
box = TwoSumHashMap()
operations = [("add", 1), ("add", 3), ("find", 4),
              ("find", 6), ("add", 3), ("find", 6)]
for operation, argument in operations:
    before = dict(box.counts)
    result = getattr(box, operation)(argument)
    print(f"{operation}({argument}): trước={before}, sau={box.counts}, trả về={result}")

add(1): trước={}, sau={1: 1}, trả về=None
add(3): trước={1: 1}, sau={1: 1, 3: 1}, trả về=None
find(4): trước={1: 1, 3: 1}, sau={1: 1, 3: 1}, trả về=True
find(6): trước={1: 1, 3: 1}, sau={1: 1, 3: 1}, trả về=False
add(3): trước={1: 1, 3: 1}, sau={1: 1, 3: 2}, trả về=None
find(6): trước={1: 1, 3: 2}, sau={1: 1, 3: 2}, trả về=True


## 19. Cú pháp `getattr` ở cell vừa rồi là gì?

Code trên dùng `getattr(box, operation)` để chọn method theo chuỗi tên `operation`. Đây **chỉ là công cụ phụ để chạy danh sách thao tác** trong ví dụ, **không phải kỹ thuật cần cho lời giải #170**.

Bạn hoàn toàn có thể viết tường minh `box.add(1)`, `box.find(4)` — dễ hiểu hơn nếu mới học.

Trong các cell trace tiếp theo, ta chủ động gọi method rõ ràng, và sẽ ghi lại trạng thái trước/sau khi hỏi. Không cần biết `getattr` để làm đúng bài.

In [18]:
box = TwoSumHashMap()
box.add(1)
box.add(3)
print("find(4):", box.find(4))
print("find(6) trước add(3):", box.find(6))
box.add(3)
print("find(6) sau add(3):", box.find(6))

find(4): True
find(6) trước add(3): False
find(6) sau add(3): True


## 20. Trace theo từng ứng viên trong `find(value)`

Ta viết hàm `trace_find_counts(counts, value)`: copy trạng thái counts rồi in:

- `x` đang xét;
- `y = value - x` cần tìm;
- `y` có trong hộp chưa;
- nếu `x == y`, tần suất đủ 2 chưa;
- vì sao trả True/False.

Mỗi lượt `find` **không sửa dictionary**. Trace được tách khỏi class submit để tránh in debug khi nộp.

In [19]:
def trace_find_counts(counts: dict, value: int) -> bool:
    print(f"find({value}), counts ban đầu={counts}")
    for x, frequency in counts.items():
        y = value - x
        exists = y in counts
        enough = (x != y or frequency >= 2)
        print(f"  x={x}, count[x]={frequency}, cần y={y}, y có? {exists}, đủ bản sao? {enough}")
        if exists and enough:
            print("  => Đã tìm thấy cặp hợp lệ, trả True")
            return True
    print("  => Không có cặp hợp lệ, trả False")
    return False

assert trace_find_counts({1: 1, 3: 1}, 6) is False

find(6), counts ban đầu={1: 1, 3: 1}
  x=1, count[x]=1, cần y=5, y có? False, đủ bản sao? True
  x=3, count[x]=1, cần y=3, y có? True, đủ bản sao? False
  => Không có cặp hợp lệ, trả False


## 21. Trace điều kiện đặc biệt bằng hai trạng thái khác nhau

- Trạng thái `{2:1}`: `find(4)` phải False.
- Trạng thái `{2:2}`: `find(4)` phải True.
- Trạng thái `{1:1, 3:1}`: `find(4)` True dù **mỗi giá trị chỉ xuất hiện một lần**, vì x và y khác nhau.

Hãy tự đoán đầu ra rồi mới chạy. Nếu một trong ba điều trên khó hiểu, quay lại mục 14.

In [20]:
assert trace_find_counts({2: 1}, 4) is False
print("---")
assert trace_find_counts({2: 2}, 4) is True
print("---")
assert trace_find_counts({1: 1, 3: 1}, 4) is True

find(4), counts ban đầu={2: 1}
  x=2, count[x]=1, cần y=2, y có? True, đủ bản sao? False
  => Không có cặp hợp lệ, trả False
---
find(4), counts ban đầu={2: 2}
  x=2, count[x]=2, cần y=2, y có? True, đủ bản sao? True
  => Đã tìm thấy cặp hợp lệ, trả True
---
find(4), counts ban đầu={1: 1, 3: 1}
  x=1, count[x]=1, cần y=3, y có? True, đủ bản sao? True
  => Đã tìm thấy cặp hợp lệ, trả True


## 22. Code nộp LeetCode chuẩn: **`class TwoSum`**, không phải `Solution`

Đây là bản gọn đầy đủ cho LeetCode #170. Không cần `from typing import List` vì API chỉ nhận từng số nguyên.

Điểm khác với hầu hết bài LeetCode dạng thuật toán là **LeetCode tạo một đối tượng `TwoSum()` rồi gọi nhiều method trên cùng đối tượng**. Khi submit, chỉ chép phần định nghĩa class ở cell này.

Phía dưới ta thử đúng chuỗi gọi chuẩn đề bài.

In [21]:
class TwoSum:
    def __init__(self):
        self.counts = {}

    def add(self, number: int) -> None:
        self.counts[number] = self.counts.get(number, 0) + 1

    def find(self, value: int) -> bool:
        for x, frequency in self.counts.items():
            y = value - x
            if y in self.counts and (x != y or frequency >= 2):
                return True
        return False

obj = TwoSum()
obj.add(1)
obj.add(3)
obj.add(5)
assert obj.find(4) is True
assert obj.find(7) is False
print("PASS: mẫu chính thức của bài 170")

PASS: mẫu chính thức của bài 170


## 23. Các test biên quan trọng: hộp rỗng, một số, trùng, 0, âm

Người mới rất dễ chỉ kiểm mẫu `[1,3,5]` rồi nghĩ bài đã xong. Nhưng ít nhất phải test:

1. Hộp trống: không có cặp → luôn False.
2. Mới add một số: không được dùng lại chính phần tử đó.
3. Hai số bằng nhau: đủ **hai bản sao** thì được.
4. Số 0: `0+0=0` cần **hai** số 0.
5. Số âm: `-3+5=2` hợp lệ.
6. Thêm số mới **sau** khi gọi find: cùng một target có thể đổi kết quả.
7. Hai đối tượng khác nhau có trạng thái độc lập.

In [22]:
obj = TwoSum()
assert obj.find(0) is False
obj.add(0)
assert obj.find(0) is False
obj.add(0)
assert obj.find(0) is True
assert obj.find(1) is False
obj.add(-3)
obj.add(5)
assert obj.find(2) is True
assert obj.find(-6) is False
obj.add(-3)
assert obj.find(-6) is True
print("PASS: biên rỗng / 0 / số âm / hai bản sao")

PASS: biên rỗng / 0 / số âm / hai bản sao


## 24. Hai đối tượng phải độc lập: lỗi dùng biến chung trong class

Nếu tạo:

```python
first = TwoSum()
second = TwoSum()
```

thì `first.add(2)` **không** khiến `second` có số 2. `self.counts` được tạo trong `__init__` của **mỗi** đối tượng.

Lỗi phổ biến: đặt `counts = {}` như một biến chung của class rồi vô tình dùng cho mọi object. Khi đó một hộp có thể “nhìn thấy” dữ liệu của hộp khác, trái yêu cầu API.

Cell dưới kiểm tra tính cách ly trạng thái.

In [23]:
first = TwoSum()
second = TwoSum()
first.add(2)
first.add(2)
assert first.find(4) is True
assert second.find(4) is False
second.add(1)
second.add(3)
assert second.find(4) is True
assert first.find(4) is True
print("PASS: hai đối tượng không chia sẻ trạng thái")

PASS: hai đối tượng không chia sẻ trạng thái


## 25. Vì sao giải thuật đúng? Hai invariant nhỏ

**Invariant 1 — sau mỗi `add(number)`:** với mọi số `x`, `self.counts[x]` bằng đúng **số lượt add(x) đã diễn ra** trên đối tượng.

- Đúng khi mới tạo vì mọi tần suất đều bằng 0 (dictionary rỗng).
- Mỗi `add(number)` chỉ tăng đúng tần suất của `number` thêm 1.
- `find` không sửa `counts`, nên invariant được bảo toàn.

**Invariant 2 — khi `find(value)` đang xét các khóa:** mọi khóa đã duyệt trước đó đều **không tạo được cặp hợp lệ**; nếu có thì hàm đã return True.

Với bất kỳ cặp hợp lệ `(x,y)`, ta sẽ gặp khóa `x`. Khi đó `y=value-x` tồn tại. Nếu khác nhau, cặp hợp lệ ngay. Nếu bằng nhau, vì là **hai lượt add khác nhau**, tần suất phải ≥ 2, nên điều kiện cũng đúng.

Ngược lại, nếu hàm trả True, nó đã xác nhận có `y` và có đủ hai bản sao nếu `x=y`. Vậy không thể trả True nhầm.

Duyệt hết không có cặp → False. Do đó `find` đúng.

## 26. Độ phức tạp Hash Map — tách `add` và `find`

Với bài thiết kế dữ liệu, **không thể chỉ ghi một Big-O chung**, vì các thao tác có thể được gọi khác số lần.

Đặt:
- `n` = tổng số lượt `add` đến hiện tại (có thể lặp giá trị).
- `k` = số giá trị **phân biệt** đang lưu (số key của dictionary), `k <= n`.

| Thao tác | Thời gian | Lý do |
|---|---|---|
| `TwoSum()` | `O(1)` | Tạo một dictionary rỗng |
| `add(number)` | `O(1)` **trung bình** | Một phép tra cứu + cập nhật Hash Map |
| `find(value)` | `O(k)` **trung bình / worst theo số khóa được xét**, nếu thao tác hash được xem là O(1) trung bình | Xét tối đa k giá trị phân biệt |
| Bộ nhớ của đối tượng | `O(k)` | Lưu một cặp `key → count` cho mỗi giá trị phân biệt |

**Ghi chú độ chặt:** Hash Map Python có thể có trường hợp va chạm khiến một lần tra cứu không đạt O(1) tuyệt đối. Vì vậy các chặn nêu trên là theo **mô hình tra cứu hash trung bình** thường dùng trong phân tích LeetCode; không khẳng định bảo đảm worst-case cho phép băm.

Nếu mọi số được thêm đều giống nhau, `n` có thể lớn nhưng `k=1`, nên mỗi `find` chỉ duyệt một khóa.

In [24]:
obj = TwoSum()
for _ in range(100):
    obj.add(7)
print("Đã add 100 lượt; số key k =", len(obj.counts), "| counts:", obj.counts)
print("find(14):", obj.find(14))
assert len(obj.counts) == 1 and obj.find(14) is True

Đã add 100 lượt; số key k = 1 | counts: {7: 100}
find(14): True


## 27. Biến thể quan trọng: nếu cần gọi `find` cực nhiều thì sao?

Hiện tại `add` nhanh, `find` phải duyệt các khóa. Có thể **đảo ưu tiên**: mỗi lần `add(new_number)`, tính ngay mọi tổng với các số đã có, rồi lưu các tổng hợp lệ vào `set`.

Ví dụ:

```text
add(1) -> sums = {}
add(3) -> tạo 1+3=4, sums={4}
add(5) -> tạo 1+5=6 và 3+5=8, sums={4,6,8}
```

Lúc gọi `find(4)`, chỉ cần kiểm `4 in sums` → `O(1)` trung bình, rất nhanh.

**Đổi lại:** mỗi `add` giờ phải tính nhiều tổng (`O(n)`) và `sums` có thể chứa tới `O(n²)` tổng khác nhau. Đây là bài học **trade-off (đánh đổi)**: tối ưu cho kiểu thao tác nào thường gặp.

## 28. Python `set`: lưu các tổng đã có, tra cứu nhanh

`set()` tạo một tập hợp giá trị **không trùng lặp**:

```python
sums = set()
sums.add(4)
sums.add(4)
```

Tập vẫn chỉ chứa `{4}` vì hai tổng bằng nhau không cần lưu hai lần. Ta chỉ hỏi có tồn tại cặp hay không.

Khác với `set` dùng để lưu **các số gốc** (có thể làm mất thông tin số bản sao), `set` lưu **các tổng của những cặp vị trí đã thực sự được tạo ra** là hợp lý: cặp 2+2 chỉ được tạo khi add số 2 lần thứ hai.

In [25]:
sums = set()
sums.add(4)
sums.add(4)
sums.add(8)
print("Các tổng hợp lệ:", sums)
print("4 in sums:", 4 in sums)
print("7 in sums:", 7 in sums)

Các tổng hợp lệ: {8, 4}
4 in sums: True
7 in sums: False


## 29. Cài đặt biến thể lưu trước mọi tổng (nhanh ở `find`)

Ở mỗi `add(number)`:

1. Duyệt toàn bộ số **đã thêm trước đó**.
2. Với từng `old`, thêm `old + number` vào `pair_sums`.
3. **Sau đó mới** thêm `number` vào danh sách `numbers`.

Thứ tự bước 3 rất quan trọng! Nếu thêm `number` vào `numbers` trước, bạn sẽ tạo tổng `number + number` bằng **chính phần tử mới với chính nó**, dù mới được thêm một lần. Đây là lỗi tương tự “kiểm tra trước, lưu sau” trong Two Sum (#1).

In [26]:
class TwoSumPairSums:
    def __init__(self):
        self.numbers = []
        self.pair_sums = set()

    def add(self, number: int) -> None:
        for old in self.numbers:
            self.pair_sums.add(old + number)
        self.numbers.append(number)

    def find(self, value: int) -> bool:
        return value in self.pair_sums

obj = TwoSumPairSums()
obj.add(2)
assert obj.find(4) is False
obj.add(2)
assert obj.find(4) is True
print("numbers:", obj.numbers, "| các tổng hợp lệ:", obj.pair_sums)

numbers: [2, 2] | các tổng hợp lệ: {4}


## 30. Trade-off ba thiết kế — so sánh theo loại thao tác

| Thiết kế | `add` | `find` | Bộ nhớ của đối tượng | Ưu tiên |
|---|---|---|---|---|
| List + thử mọi cặp | `O(1)` amortized | `O(n²)` | `O(n)` | Dễ nghĩ, học vòng lặp |
| **Dict đếm tần suất** | **`O(1)` trung bình** | **`O(k)` trung bình** | **`O(k)`** | Nhiều `add`, ít/lẫn `find` |
| Set các tổng đã từng tạo | `O(n)` trung bình | **`O(1)` trung bình** | `O(n²)` worst | Nhiều `find`, chấp nhận add chậm / tốn nhớ |

`n` là số lượt add, `k` là số giá trị phân biệt.

**Chú ý:** trong phương án “set tổng”, `find` chỉ kiểm membership; nhưng `add` có thể tăng thời gian tổng cộng đến `O(n²)` sau n lần add, và set có thể rất lớn. Vì vậy không phải cứ `find O(1)` là luôn tốt hơn.

Trong bài #170, **dictionary đếm tần suất** là lựa chọn gọn, dễ chứng minh và cân bằng tốt.

## 31. Kiểm thử so sánh 3 phiên bản bằng các lệnh giống nhau

Thay vì chỉ so sánh đáp án cuối, kiểm tra **từng lần `find` sau từng lượt `add`**, để bắt lỗi mất trạng thái.

Ta sẽ tạo ba hộp cùng nhận các lệnh và kiểm tất cả kết quả `find` giống nhau. Một `find` sai sẽ báo `AssertionError` ngay tại thao tác đó.

In [27]:
operations = [
    ("find", 4), ("add", 1), ("find", 2),
    ("add", 3), ("find", 4), ("find", 6),
    ("add", 3), ("find", 6), ("add", -1),
    ("find", 0), ("add", 0), ("find", -1),
]
models = [TwoSumListBrute(), TwoSumHashMap(), TwoSumPairSums()]
for name, arg in operations:
    if name == "add":
        for model in models:
            model.add(arg)
    else:
        answers = [model.find(arg) for model in models]
        assert answers[0] == answers[1] == answers[2], (name, arg, answers)
        print(f"find({arg}) -> {answers[0]}")
print("PASS: 3 phiên bản cho kết quả nhất quán từng lần gọi")

find(4) -> False
find(2) -> False
find(4) -> True
find(6) -> False
find(6) -> True
find(0) -> True
find(-1) -> True
PASS: 3 phiên bản cho kết quả nhất quán từng lần gọi


## 32. Kiểm thử ngẫu nhiên có hạt giống (reproducible)

Chúng ta tạo nhiều dãy **thêm / hỏi xen kẽ**, dùng các số nhỏ âm và dương. Cách này giúp phát hiện lỗi khi chương trình từng gọi `find`, sau đó `add`, rồi lại `find`.

`random.Random(170)` cho kết quả ngẫu nhiên lặp lại được giữa các lần chạy. Khi không có cặp, ba cách phải cùng trả False; khi có cặp thì cùng True.

Mỗi ca chỉ có tối đa vài chục thao tác để brute force chạy nhanh; không kiểm brute force trên hàng chục nghìn phần tử.

In [28]:
import random
rng = random.Random(170)
for case in range(100):
    boxes = [TwoSumListBrute(), TwoSumHashMap(), TwoSumPairSums(), TwoSum()]
    for step in range(40):
        if rng.random() < 0.6:
            number = rng.randint(-10, 10)
            for box in boxes:
                box.add(number)
        else:
            target = rng.randint(-20, 20)
            answers = [box.find(target) for box in boxes]
            assert all(answer == answers[0] for answer in answers), (case, step, target, answers)
print("PASS: 100 chuỗi × 40 thao tác; 4 thiết kế nhất quán ở mọi lần find")

PASS: 100 chuỗi × 40 thao tác; 4 thiết kế nhất quán ở mọi lần find


## 33. Các lỗi thường gặp: sai → tại sao → sửa

**Sai 1 — Chỉ lưu `set(numbers)` rồi hỏi `y in seen`.** Nếu mới `add(2)` một lần, `find(4)` không được True. **Sửa:** đếm tần suất bằng dict hoặc chỉ tạo các tổng khi có hai lần add thực sự.

**Sai 2 — Không có `self` / khai báo `counts = {}` trong từng lần `add`.** Dữ liệu bị quên sau mỗi lời gọi. **Sửa:** tạo `self.counts = {}` **trong `__init__`**.

**Sai 3 — Chép `class Solution` từ bài khác.** API #170 yêu cầu `class TwoSum` với `add`, `find`. **Sửa:** tuân đúng chữ ký.

**Sai 4 — `return False` đặt ngay trong vòng `for`.** Chưa thử các khóa khác mà đã kết luận. **Sửa:** chỉ `return False` **sau khi for kết thúc**.

**Sai 5 — Dùng lại cùng một lượt thêm.** Với `x == y`, phải `counts[x] >= 2`.

**Sai 6 — `add` trả list/tần suất thay vì `None`.** API chỉ cần cập nhật trạng thái, không yêu cầu trả dữ liệu.

**Sai 7 — Sửa counts trong `find`.** Hỏi không được làm mất hoặc thêm số. **Sửa:** chỉ đọc trong `find`.

**Sai 8 — Biến thể precompute sums lưu `number` trước khi tạo cặp.** Tự ghép chính mình; **phải tạo tổng với số cũ trước khi append số mới**.

In [29]:
# Minh họa lỗi 'return False' quá sớm mà không làm hỏng lời giải chuẩn.
counts = {1: 1, 3: 1}
value = 4
first_x = next(iter(counts))
print("Nếu chỉ nhìn khóa đầu:", first_x, "sẽ chưa đủ để kết luận mọi cặp.")
print("Kết quả tìm đúng:", trace_find_counts(counts, value))

Nếu chỉ nhìn khóa đầu: 1 sẽ chưa đủ để kết luận mọi cặp.
find(4), counts ban đầu={1: 1, 3: 1}
  x=1, count[x]=1, cần y=3, y có? True, đủ bản sao? True
  => Đã tìm thấy cặp hợp lệ, trả True
Kết quả tìm đúng: True


## 34. Mẹo nhận dạng dạng bài “thiết kế dữ liệu”

Các dấu hiệu cho thấy đây **không** chỉ là hàm tính kết quả một lần:

1. Đề viết “Design a data structure” / “Implement the class”.
2. Có `constructor` / `__init__` và **nhiều** method.
3. Kết quả của `find` phụ thuộc **lịch sử gọi `add`**.
4. Có yêu cầu phân tích **độ phức tạp từng thao tác**, chứ không chỉ toàn chương trình.
5. Có thể tối ưu `add` hoặc `find`, nhưng phải xét tần suất sử dụng thực tế.

Hãy luôn tự hỏi: “Đối tượng cần nhớ điều gì tối thiểu để trả lời câu hỏi tiếp theo?” Ở #170, câu trả lời là **giá trị và số lần nó đã được thêm**.

## 35. Những biến thể thực tế có thể gặp

**Biến thể A — `find(value)` đồng thời trả về cặp `(x,y)`:** khi tìm thấy, trả cặp giá trị thay vì True. Cần vẫn bảo đảm hai lượt `add` khác nhau.

**Biến thể B — `remove(number)`:** giảm tần suất của số trong dictionary; nếu tần suất về 0 thì xóa khóa. Nhưng nếu đang dùng set các tổng đã tạo thì remove khó hơn, vì có tổng có thể được tạo bởi nhiều cặp.

**Biến thể C — `countPairs(value)`:** đếm số cặp vị trí khác nhau. Chỉ kiểm có/không không đủ; lúc đó tần suất `count[x]` rất có ích. Phải tránh đếm trùng `(x,y)` và `(y,x)`.

**Biến thể D — Dữ liệu đến theo luồng:** theo thời gian, các lượt add tiếp tục đến. Kỹ thuật lưu trạng thái còn xuất hiện ở log, bộ đếm sự kiện, cửa sổ trượt, dữ liệu streaming.

## 36. Bài tập tự luyện — 6 nhiệm vụ tăng dần

Hãy giữ các bài dưới là code TODO chưa hoàn thiện và **không gọi tự động**. Khi làm xong, mở dấu `#` ở `assert` tương ứng.

Lộ trình:
1. Đọc trạng thái & đoán output.
2. Tính tay và trace thao tác.
3. Viết lại class bằng list brute force.
4. Viết lại class bằng dictionary.
5. Biến thể trả cặp giá trị.
6. Mini challenge hỗ trợ `remove` (chú ý tần suất).

Phần gợi ý có **3 cấp** và đáp án nằm ở **cuối notebook**.

### Bài tập 1 — Đoán output của object / dict / set

Không chạy trước:

```python
x = {}
x[2] = x.get(2, 0) + 1
x[2] = x.get(2, 0) + 1
print(x)
```

1. `x` là gì sau hai lần tăng?
2. `2 in x` bằng gì?
3. `x.get(9,0)` bằng gì?
4. `len(set([2,2]))` bằng gì?
5. `len([2,2])` bằng gì?

Giải thích vì sao hai câu cuối khác nhau và hệ quả với `find(4)`.

In [30]:
# Bài 1: chỉ dùng để tự kiểm sau khi đã dự đoán.
# x = {}
# x[2] = x.get(2, 0) + 1
# x[2] = x.get(2, 0) + 1
# print(x, 2 in x, x.get(9, 0), len(set([2, 2])), len([2, 2]))
pass

### Bài tập 2 — Trace các lệnh theo bảng

Bắt đầu `TwoSum()` rỗng, lần lượt:

```text
find(8), add(4), find(8), add(4), find(8), add(-2), find(2)
```

Tự lập bảng có các cột:

| Lệnh | counts trước | Cặp đang kiểm | counts sau | return |
|---|---|---|---|---|

**Đặc biệt:** vì sao hai lần `find(8)` lần đầu và lần thứ hai có thể đều False dù đã có `4`? Lần thứ ba vì sao là True?

In [31]:
# Chỉ bỏ dấu # để tự đối chiếu sau khi làm bảng:
# obj = TwoSum()
# print(obj.find(8))
# obj.add(4)
# print(obj.find(8))
# obj.add(4)
# print(obj.find(8))
# obj.add(-2)
# print(obj.find(2))
pass

### Bài tập 3 — Tự viết class brute force `ExerciseTwoSumList`

Chỉ được lưu `self.numbers` là list. `add` dùng `append`; `find` dùng hai vòng `for` với `i < j`.

Sau khi viết, tự test:

```python
obj = ExerciseTwoSumList()
obj.add(2)
assert obj.find(4) is False
obj.add(2)
assert obj.find(4) is True
```

In [32]:
class ExerciseTwoSumList:
    def __init__(self):
        # TODO: tạo list numbers thuộc đối tượng.
        pass

    def add(self, number: int) -> None:
        # TODO: thêm number vào list.
        pass

    def find(self, value: int) -> bool:
        # TODO: duyệt i < j và kiểm tổng.
        pass

# Chỉ test SAU KHI tự hoàn thiện class.

### Bài tập 4 — Tự viết `ExerciseTwoSumHash` bằng dictionary

Yêu cầu:
1. Dữ liệu được lưu vào `self.counts = {}`.
2. `add` tăng đúng số lần của từng số.
3. `find` tìm `y = value - x` từ các **key** có trong counts.
4. Nếu `x == y`, phải `counts[x] >= 2`.
5. Chạy đúng dù có số âm và số 0.

In [33]:
class ExerciseTwoSumHash:
    def __init__(self):
        # TODO: self.counts = ...
        pass

    def add(self, number: int) -> None:
        # TODO: tăng tần suất của number.
        pass

    def find(self, value: int) -> bool:
        # TODO: duyệt các key, tính needed, kiểm tồn tại và bản sao.
        pass

# Tự kiểm sau khi viết:
# obj = ExerciseTwoSumHash()
# obj.add(0); obj.add(0)
# assert obj.find(0) is True

### Bài tập 5 — Biến thể: trả **cặp giá trị** hoặc `None`

Viết `exercise_find_pair(counts, value)`:

- Nếu tồn tại hai giá trị trong hộp có tổng `value`, trả `(x,y)` (thứ tự không quan trọng).
- Nếu không có, trả `None`.

Đặc biệt:
- `{2:1}`, target 4 → `None`.
- `{2:2}`, target 4 → `(2,2)`.
- `{1:1,3:1}`, target 4 → `(1,3)` hoặc `(3,1)`.

Chỉ sử dụng các **khóa và tần suất** trong dict, không cần danh sách gốc.

In [34]:
def exercise_find_pair(counts: dict, value: int):
    # TODO: duyệt x trong counts; y = value-x;
    # nếu y có và đủ số bản sao, return (x,y), ngược lại None.
    pass

# assert exercise_find_pair({2: 2}, 4) == (2, 2)

### Bài tập 6 — Mini challenge: hỗ trợ thêm `remove(number)`

Thiết kế `ExerciseTwoSumWithRemove` dựa trên dictionary, ngoài hai method `add` / `find` còn có `remove(number) -> bool`:

- Nếu `number` chưa tồn tại → trả False và không sửa gì.
- Nếu có → xóa **một bản sao**, trả True.
- Nếu tần suất giảm xuống 0 → xóa hẳn key để `find` không nhìn thấy số đó nữa.

Ví dụ: `add(2)` hai lần → `find(4)` True; `remove(2)` một lần → `find(4)` False; `remove(2)` lần nữa → dictionary không còn khóa 2.

Đây là **mở rộng tự luyện**, không phải method trong API nộp LeetCode #170.

In [35]:
class ExerciseTwoSumWithRemove:
    def __init__(self):
        # TODO: tạo dict counts.
        pass

    def add(self, number: int) -> None:
        # TODO: tăng tần suất.
        pass

    def find(self, value: int) -> bool:
        # TODO: logic giống bài 4.
        pass

    def remove(self, number: int) -> bool:
        # TODO: nếu không có trả False.
        # TODO: nếu có giảm count; bằng 0 thì xóa key; trả True.
        pass

# Chỉ test sau khi tự hoàn thành.

## 37. Gợi ý 3 cấp cho từng bài — đọc từ dễ tới gần code

| Bài | Mức 1 (hướng nghĩ) | Mức 2 (cú pháp) | Mức 3 (gần code) |
|---|---|---|---|
| 1 | `dict` giữ tần suất | `.get(x,0)`; `in` | Hai lần tăng 2 tạo `{2:2}` |
| 2 | `find` không sửa counts | `x+y=target` | 4+4 chỉ được khi đã add 4 hai lần |
| 3 | Thử mọi **cặp vị trí** | `range(i+1,n)` | `if self.numbers[i]+self.numbers[j]==value` |
| 4 | Đếm để tránh dùng một phần tử hai lần | `for x,freq in self.counts.items()` | `if needed in counts and (needed != x or freq >= 2)` |
| 5 | Khi thấy cặp, trả tuple | `return (x,needed)` | Không có cặp mới `return None` sau for |
| 6 | Xóa đúng **một bản sao** | `del self.counts[number]` | Nếu `counts[number]==1` thì xóa khóa; nếu >1 thì giảm |

Đừng xem đáp án ngay. Với mỗi TODO, hãy trả lời được câu hỏi: **Tại sao dòng đó phải nằm đúng vị trí ấy?**

## 38. Đáp án Bài 1–2 (đọc sau khi tự làm)

**Bài 1:**
1. `x = {2:2}`.
2. `2 in x` là True (đó là một key).
3. `x.get(9,0)` là 0.
4. `len(set([2,2]))` là 1.
5. `len([2,2])` là 2.

`set` xóa sự khác biệt giữa một và hai bản sao — lý do dùng dict đếm tần suất.

**Bài 2:**

| Thao tác | counts sau | Kết quả |
|---|---|---|
| `find(8)` | `{}` | False |
| `add(4)` | `{4:1}` | None |
| `find(8)` | `{4:1}` | False (thiếu số 4 thứ hai) |
| `add(4)` | `{4:2}` | None |
| `find(8)` | `{4:2}` | True (4+4) |
| `add(-2)` | `{4:2,-2:1}` | None |
| `find(2)` | `{4:2,-2:1}` | True (4 + -2) |

In [36]:
obj = TwoSum()
assert obj.find(8) is False
obj.add(4)
assert obj.find(8) is False
obj.add(4)
assert obj.find(8) is True
obj.add(-2)
assert obj.find(2) is True
print("PASS Bài 2: trace thao tác")

PASS Bài 2: trace thao tác


### Đáp án Bài 3 — Brute force bằng list

Đây là lời giải cơ sở: `find` thử các cặp chỉ số khác nhau, nên không dùng lại một phần tử. Kỹ thuật này hữu ích khi đối chiếu với phương pháp tối ưu.

In [37]:
class AnswerTwoSumList:
    def __init__(self):
        self.numbers = []

    def add(self, number: int) -> None:
        self.numbers.append(number)

    def find(self, value: int) -> bool:
        for i in range(len(self.numbers)):
            for j in range(i + 1, len(self.numbers)):
                if self.numbers[i] + self.numbers[j] == value:
                    return True
        return False

obj = AnswerTwoSumList()
obj.add(2)
assert obj.find(4) is False
obj.add(2)
assert obj.find(4) is True
print("PASS Bài 3")

PASS Bài 3


### Đáp án Bài 4 — Dictionary đếm tần suất

Mỗi key là **giá trị số**, mỗi value là **số bản sao** đã thêm. `find` chỉ cần duyệt key và tính số còn thiếu. Bản sao thứ hai chỉ cần khi hai số trong cặp **bằng nhau**.

In [38]:
class AnswerTwoSumHash:
    def __init__(self):
        self.counts = {}

    def add(self, number: int) -> None:
        self.counts[number] = self.counts.get(number, 0) + 1

    def find(self, value: int) -> bool:
        for x, frequency in self.counts.items():
            needed = value - x
            if needed in self.counts and (needed != x or frequency >= 2):
                return True
        return False

obj = AnswerTwoSumHash()
obj.add(0)
assert obj.find(0) is False
obj.add(0)
assert obj.find(0) is True
obj.add(-5); obj.add(7)
assert obj.find(2) is True
print("PASS Bài 4")

PASS Bài 4


### Đáp án Bài 5 — Trả cặp giá trị

Nếu không tìm thấy cặp nào, hàm trả `None`. Nếu tìm thấy, trả tuple hai số. Hãy chú ý nhánh `x == y` vẫn không được bỏ.

In [39]:
def answer_find_pair(counts: dict, value: int):
    for x, frequency in counts.items():
        y = value - x
        if y in counts and (x != y or frequency >= 2):
            return (x, y)
    return None

assert answer_find_pair({2: 1}, 4) is None
assert answer_find_pair({2: 2}, 4) == (2, 2)
assert set(answer_find_pair({1: 1, 3: 1}, 4)) == {1, 3}
print("PASS Bài 5")

PASS Bài 5


### Đáp án Bài 6 — Bổ sung `remove` đúng cách

Nếu đang có hai bản sao 2, xóa một lần thì vẫn còn 2 trong dictionary với tần suất 1. Nhưng lúc này `find(4)` phải False.

Nếu tần suất còn 1 và lại remove, hãy **xóa khóa** (`del ...`) thay vì để giá trị 0 trong dictionary. Nếu để khóa với tần suất 0, điều kiện `needed in counts` sẽ nhầm rằng giá trị vẫn tồn tại.

In [40]:
class AnswerTwoSumWithRemove:
    def __init__(self):
        self.counts = {}

    def add(self, number: int) -> None:
        self.counts[number] = self.counts.get(number, 0) + 1

    def find(self, value: int) -> bool:
        for x, frequency in self.counts.items():
            y = value - x
            if y in self.counts and (x != y or frequency >= 2):
                return True
        return False

    def remove(self, number: int) -> bool:
        if number not in self.counts:
            return False
        if self.counts[number] == 1:
            del self.counts[number]
        else:
            self.counts[number] -= 1
        return True

obj = AnswerTwoSumWithRemove()
assert obj.remove(2) is False
obj.add(2); obj.add(2)
assert obj.find(4) is True
assert obj.remove(2) is True
assert obj.find(4) is False
assert obj.remove(2) is True
assert obj.remove(2) is False
assert 2 not in obj.counts
print("PASS Bài 6")

PASS Bài 6


## 39. Tổng kết, code template và checklist cuối

**Câu nhớ nhanh:** “Hộp cần nhớ **mỗi số đã được thêm bao nhiêu lần**. Khi hỏi tổng `value`, chọn `x`, tìm `needed = value - x`. Nếu hai số bằng nhau, cần ít nhất hai bản sao.”

**Template tư duy cho bài thiết kế cấu trúc dữ liệu:**

1. Liệt kê API: constructor, method cập nhật, method truy vấn.
2. Xác định thông tin phải sống **qua nhiều lần gọi** → thuộc tính `self...`.
3. Viết cách trực giác cho đúng trước (ở đây: list + 2 vòng for).
4. Tìm phần việc bị lặp lại → dùng dict để tra phần tử còn thiếu.
5. Xác định trường hợp trùng (`x == needed`) và cách bảo đảm **hai bản sao**.
6. Phân tích **độ phức tạp từng method**, không gom chung.

**Checklist trước khi submit:**
- [ ] Tên class là **`TwoSum`**, **không** phải `Solution`?
- [ ] Có `__init__(self)`, `add(self, number)`, `find(self, value)` đúng chữ ký?
- [ ] `self.counts` được khởi tạo cho từng object?
- [ ] `add` tăng tần suất và không bắt buộc trả kết quả?
- [ ] `find` không thay đổi counts và trả bool?
- [ ] `needed = value - x` và điều kiện `x != needed or count >= 2` đúng?
- [ ] `return False` nằm **sau** vòng for?
- [ ] Đã test hộp trống, một số, hai số trùng, số 0, số âm, các lượt add/find xen kẽ và hai object độc lập?

Nếu bạn tự xây lại được class bằng cả `list` lẫn `dict`, và giải thích được vì sao `find(4)` khác nhau giữa một và hai lần `add(2)`, bạn đã nắm bản chất bài #170.